# Knee MRI: error analysis of the baseline OOF predictions
CPU only (Accelerator: None). Pandas + scikit-learn, takes a minute.
**Inputs to attach:** baseline training output (`knee-mri-diagnosis-training`, has `oof_all.csv`), caching output (`knee-mri-diagnosis-caching`, has `cache_index.csv`), the competition data (for `train.csv`, optional).
Run top to bottom. Cell 1 prints what it found and the column names; if it refuses to guess, set the `*_OVERRIDE` variables.
Untested on the real files: it was only tested on synthetic data.

In [1]:
# 0. config
import os, glob, re
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 120)

IN = "/kaggle/input"
TRAIN_NB = f"{IN}/notebooks/theokorir/knee-mri-diagnosis-training"
CACHE_NB = f"{IN}/notebooks/theokorir/knee-mri-diagnosis-caching"
DATA     = f"{IN}/competitions/rsna-knee-abnormality-detection"
OUT = "/kaggle/working/error_analysis"; os.makedirs(OUT, exist_ok=True)

# overrides (leave None unless cell 1 asks)
OOF_OVERRIDE = None          # path to oof_all.csv
META_OVERRIDE = None         # path to cache_index.csv (site / vendor columns)
TRAIN_CSV_OVERRIDE = None    # path to competition train.csv (report text)
PRED_PREFIX_OVERRIDE = None  # e.g. "p_"
GROUP_COLS_OVERRIDE = None   # e.g. ["vendor", "site_proxy"]
LABEL = "ACL"                # label to inspect in detail (cells 5-6)

for d in (TRAIN_NB, CACHE_NB, DATA):
    print(d, "->", sorted(os.listdir(d))[:12] if os.path.isdir(d) else "MISSING")

/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training -> ['__huggingface_repos__.json', '__notebook__.ipynb', '__output__.json', '__results__.html', 'custom.css', 'models_g0', 'models_g0.log', 'models_g1', 'models_g1.log', 'oof_all.csv']
/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching -> ['__notebook__.ipynb', '__output__.json', '__results__.html', '__results___files', 'cache', 'custom.css', 'labels_soft.csv', 'reliability.csv']
/kaggle/input/competitions/rsna-knee-abnormality-detection -> ['sample_submission.csv', 'test.csv', 'test_series', 'test_series.csv', 'train.csv', 'train_series', 'train_series.csv']


In [2]:
# 1. load OOF predictions and find columns
def pick(cands, override=None, what=""):
    if override: return override
    found = [c for c in cands if os.path.exists(c)]
    print(what, "candidates found:", found)
    if len(found) != 1:
        raise SystemExit(f"{what}: {len(found)} candidates; set the override variable")
    return found[0]

oof_c = glob.glob(f"{TRAIN_NB}/**/oof_all.csv", recursive=True)
OOF = pick(oof_c, OOF_OVERRIDE, "oof_all.csv")
oof = pd.read_csv(OOF)
print(oof.shape); print(list(oof.columns))

ID = next((c for c in oof.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
assert ID, "no study id column; see printed columns"
ycols = [c for c in oof.columns if c.startswith("y_")]
labels = [c[2:] for c in ycols]
assert len(labels) == 12, f"expected 12 y_ columns, got {len(labels)}: {ycols}"
if PRED_PREFIX_OVERRIDE: PP = PRED_PREFIX_OVERRIDE
else:
    PP = next((p for p in ["p_", "pred_", "prob_", "pr_", "s_", "score_"] if all(p + l in oof.columns for l in labels)), None)
    if PP is None:
        raise SystemExit("could not find prediction columns; set PRED_PREFIX_OVERRIDE. Non-y columns: "
                         + str([c for c in oof.columns if not c.startswith("y")]))
print("id:", ID, "| prediction prefix:", PP, "| fold column:", "fold" if "fold" in oof.columns else "none")
print("rows with predictions:", oof[PP + labels[0]].notna().sum(), "of", len(oof))

oof_all.csv candidates found: ['/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv']
(4407, 28)
['StudyInstanceUID', 'fold', 'site_proxy', 'is_gold', 'p_ACL', 'y_ACL', 'p_MCL', 'y_MCL', 'p_Medial Meniscus', 'y_Medial Meniscus', 'p_Lateral Meniscus', 'y_Lateral Meniscus', 'p_Medial OA', 'y_Medial OA', 'p_Lateral OA', 'y_Lateral OA', 'p_PF OA', 'y_PF OA', 'p_Effusion', 'y_Effusion', 'p_Synovitis', 'y_Synovitis', "p_Baker's", "y_Baker's", 'p_Contusion', 'y_Contusion', 'p_Fracture', 'y_Fracture']
id: StudyInstanceUID | prediction prefix: p_ | fold column: fold
rows with predictions: 4407 of 4407


In [3]:
# 2. metadata (site / vendor / language), merged on study id
meta_c = [f"{CACHE_NB}/cache_index.csv", f"{CACHE_NB}/cache/cache_index.csv"] + glob.glob(f"{CACHE_NB}/*/cache_index.csv")
meta_c = sorted(set(meta_c))
META = pick(meta_c, META_OVERRIDE, "cache_index.csv") if (META_OVERRIDE or any(os.path.exists(c) for c in meta_c)) else None
G = []
if META:
    meta = pd.read_csv(META)
    print(meta.shape); print(list(meta.columns))
    mid = next((c for c in meta.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
    assert mid, "no study id in metadata"
    def vend(s):
        s = str(s).lower()
        if "siemens" in s: return "Siemens"
        if "philips" in s: return "Philips"
        if "toshiba" in s or "canon" in s: return "Toshiba"
        if s.startswith("ge") or "general electric" in s: return "GE"
        return str(s)
    for c in meta.columns:
        if re.search("vendor|manufact", c, re.I): meta[c] = meta[c].map(vend)
    guess = [c for c in meta.columns if c != mid and re.search("vendor|manufact|site|lang|scanner|model|field", c, re.I)
             and 2 <= meta[c].nunique() <= 80]
    G = GROUP_COLS_OVERRIDE or guess
    print("grouping columns:", G)
    keep = [mid] + [c for c in G if c in meta.columns]
    oof = oof.merge(meta[keep].rename(columns={mid: ID}).drop_duplicates(ID), on=ID, how="left")
else:
    print("no metadata file found")
# language usually lives in train.csv, not cache_index.csv: merge it too if present
_t = TRAIN_CSV_OVERRIDE or f"{DATA}/train.csv"
if os.path.exists(_t):
    _tr = pd.read_csv(_t, nrows=5)
    _tid = next((c for c in _tr.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
    _lang = [c for c in _tr.columns if re.search("lang", c, re.I) and c not in oof.columns]
    if _tid and _lang:
        _full = pd.read_csv(_t, usecols=[_tid] + _lang).rename(columns={_tid: ID}).drop_duplicates(ID)
        oof = oof.merge(_full, on=ID, how="left"); G = list(G) + _lang
        print("added language column(s) from train.csv:", _lang)
if not G: print("no grouping columns found: cell 4 will print nothing")

cache_index.csv candidates found: ['/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache/cache_index.csv']
(4407, 16)
['StudyInstanceUID', 'ok', 'site_proxy', 'err', 'present_0', 'plane_0', 'fluid_0', 'fat_0', 'present_1', 'plane_1', 'fluid_1', 'fat_1', 'present_2', 'plane_2', 'fluid_2', 'fat_2']
grouping columns: ['site_proxy']


In [4]:
# 3. per-label AUC on confident labels (same rule as the earlier yardstick) + pair loss per study
def conf(y): return (y >= 0.9) | (y <= 0.1)

def pair_loss(yb, s):
    """Per-study share of the ranking error. For a positive: fraction of negatives scored at or above it.
    For a negative: fraction of positives scored at or below it. Mean over positives = mean over negatives = 1-AUC."""
    yb = np.asarray(yb); s = np.asarray(s, float)
    pos, neg = np.sort(s[yb == 1]), np.sort(s[yb == 0])
    out = np.zeros(len(s))
    for i, (yy, ss) in enumerate(zip(yb, s)):
        if yy == 1:
            below = np.searchsorted(neg, ss, "left") + 0.5 * (np.searchsorted(neg, ss, "right") - np.searchsorted(neg, ss, "left"))
            out[i] = 1 - below / len(neg)
        else:
            below = np.searchsorted(pos, ss, "left") + 0.5 * (np.searchsorted(pos, ss, "right") - np.searchsorted(pos, ss, "left"))
            out[i] = below / len(pos)
    return out

rows, long = [], []
for l in labels:
    d = oof[[ID, "y_" + l, PP + l] + [c for c in G if c in oof.columns]].dropna(subset=["y_" + l, PP + l])
    d = d[conf(d["y_" + l])].copy()
    d["yb"] = (d["y_" + l] >= 0.9).astype(int)
    if d["yb"].nunique() < 2: continue
    d["label"] = l; d["score"] = d[PP + l]
    d["pct"] = d["score"].rank(pct=True)        # 1.0 = highest score for this label
    d["pair_loss"] = pair_loss(d["yb"], d["score"])
    a = roc_auc_score(d["yb"], d["score"])
    rows.append(dict(label=l, n=len(d), positives=int(d["yb"].sum()), AUC=round(a, 3), lost=round(1 - a, 3)))
    long.append(d[[ID, "label", "yb", "y_" + l, "score", "pct", "pair_loss"] + [c for c in G if c in d.columns]].rename(columns={"y_" + l: "y_raw"}))
tab = pd.DataFrame(rows).sort_values("lost", ascending=False)
err = pd.concat(long, ignore_index=True)
print(tab.to_string(index=False))
print("\nmacro AUC (pooled OOF over all rows with predictions):", round(tab["AUC"].mean(), 4),
      "| expect ~0.76 pooled for the full baseline; ~0.78 on fold 0 only")
if "fold" in oof.columns:
    print("per-fold macro AUC:")
    for f, dd in oof.groupby("fold"):
        aucs = []
        for l in labels:
            x = dd[["y_" + l, PP + l]].dropna(); x = x[conf(x["y_" + l])]
            yb = (x["y_" + l] >= 0.9).astype(int)
            if yb.nunique() == 2: aucs.append(roc_auc_score(yb, x[PP + l]))
        print("  fold", f, round(float(np.mean(aucs)), 4) if aucs else "n/a")
tab.to_csv(f"{OUT}/per_label_auc.csv", index=False)

           label    n  positives   AUC  lost
           PF OA 4242       1589 0.697 0.303
             MCL 4300        542 0.732 0.268
      Lateral OA 4234        714 0.734 0.266
Lateral Meniscus 4128        618 0.738 0.262
       Contusion 4101        614 0.751 0.249
         Baker's 4350       1298 0.752 0.248
 Medial Meniscus 4099       1669 0.763 0.237
        Fracture 4321        215 0.778 0.222
             ACL 4247        634 0.779 0.221
       Synovitis 4392       1011 0.780 0.220
        Effusion 3734       2049 0.822 0.178
       Medial OA 4275       1459 0.824 0.176

macro AUC (pooled OOF over all rows with predictions): 0.7625 | expect ~0.76 pooled for the full baseline; ~0.78 on fold 0 only
per-fold macro AUC:
  fold 0 0.7795
  fold 1 0.7557
  fold 2 0.7743
  fold 3 0.7663
  fold 4 0.7691


In [5]:
# 4. where do the errors cluster? AUC by vendor / site / language group (groups with >=30 studies; labels need >=5 of each class)
def macro_auc(sub):
    aucs = []
    for l, d in sub.groupby("label"):
        if d["yb"].sum() >= 5 and (1 - d["yb"]).sum() >= 5:
            aucs.append(roc_auc_score(d["yb"], d["score"]))
    return (np.mean(aucs) if aucs else np.nan), len(aucs)

overall = macro_auc(err)[0]
print("overall macro AUC on this table:", round(overall, 4))
for g in [c for c in G if c in err.columns]:
    rows = []
    for v, sub in err.groupby(g):
        n_st = sub[ID].nunique()
        if n_st < 30: continue
        m, k = macro_auc(sub)
        rows.append(dict(group=v, studies=n_st, labels_scored=k, macro_AUC=round(m, 3), vs_overall=round(m - overall, 3)))
    if rows:
        print(f"\n== by {g} ==")
        print(pd.DataFrame(rows).sort_values("macro_AUC").to_string(index=False))
print("\nCaveat: differences under ~0.02 are inside the seed/sampling noise of this project; a vendor effect is not a site effect (confounded).")

overall macro AUC on this table: 0.7624

Caveat: differences under ~0.02 are inside the seed/sampling noise of this project; a vendor effect is not a site effect (confounded).


In [6]:
# 5. how concentrated is the error for one label, and which studies are the worst?
d = err[err["label"] == LABEL].copy()
tot = d["pair_loss"].sum()
for yb, name in [(1, "positives"), (0, "negatives")]:
    x = d[d["yb"] == yb].sort_values("pair_loss", ascending=False)
    k = max(1, int(0.1 * len(x)))
    print(f"{LABEL}: worst 10% of {name} ({k} studies) carry {x['pair_loss'].head(k).sum() / x['pair_loss'].sum():.0%} of that class's ranking error")
cols = [ID, "yb", "y_raw", "score", "pct", "pair_loss"] + [c for c in G if c in d.columns]
print("\nPositives the model scored lowest (missed findings, or wrong labels):")
print(d[d.yb == 1].sort_values("score").head(15)[cols].to_string(index=False))
print("\nNegatives the model scored highest (false alarms, or wrong labels):")
print(d[d.yb == 0].sort_values("score", ascending=False).head(15)[cols].to_string(index=False))
err.sort_values(["label", "pair_loss"], ascending=[True, False]).groupby("label").head(200).to_csv(f"{OUT}/top_errors_per_label.csv", index=False)

ACL: worst 10% of positives (63 studies) carry 36% of that class's ranking error
ACL: worst 10% of negatives (361 studies) carry 30% of that class's ranking error

Positives the model scored lowest (missed findings, or wrong labels):
                                                StudyInstanceUID  yb  y_raw    score      pct  pair_loss
1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960   1   0.95 0.116625 0.001884   0.998063
1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679   1   0.95 0.127986 0.006593   0.992804
1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636   1   0.95 0.130450 0.008006   0.991420
1.2.826.0.1.3680043.8.498.93655619137163635289231840294252859840   1   0.95 0.136429 0.016718   0.981456
1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449   1   0.95 0.143004 0.026607   0.970108
1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652   1   0.95 0.146021 0.033435   0.962358
1.2.826.0.1.3680043.8.498.59047

In [7]:
# 6. read the reports behind the worst errors: is it the label or the model?
tcs = [TRAIN_CSV_OVERRIDE] if TRAIN_CSV_OVERRIDE else [f"{DATA}/train.csv"]
if not os.path.exists(tcs[0]):
    print("train.csv not found; attach the competition data or set TRAIN_CSV_OVERRIDE"); raise SystemExit
tr = pd.read_csv(tcs[0])
print(list(tr.columns))
tid = next((c for c in tr.columns if c.lower() in ("studyinstanceuid", "study_id", "studyuid")), None)
obj = [c for c in tr.columns if not pd.api.types.is_numeric_dtype(tr[c]) and c != tid]
rep = max(obj, key=lambda c: tr[c].astype(str).str.len().mean())   # longest text column = the report
print("study id:", tid, "| report column (guess):", rep)

N = 20
worst = pd.concat([d[d.yb == 1].sort_values("score").head(N), d[d.yb == 0].sort_values("score", ascending=False).head(N)])
rev = worst[[ID, "yb", "y_raw", "score"]].merge(tr[[tid, rep]].rename(columns={tid: ID, rep: "report"}), on=ID, how="left")
rev["report"] = rev["report"].astype(str).str.replace(r"\s+", " ", regex=True).str.slice(0, 500)
rev["my_verdict"] = ""     # fill in: label_wrong / ambiguous / real_miss
rev.insert(0, "label", LABEL)
rev.to_csv(f"{OUT}/review_{LABEL.replace(' ', '_')}.csv", index=False)
for _, r in rev.iterrows():
    print(f"[{'POS' if r.yb else 'NEG'}] y={r.y_raw:.2f} score={r.score:.3f}\n   {r.report}\n")
print("Saved review sheet. Mark each case label_wrong / ambiguous / real_miss, then count them.")

['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
study id: StudyInstanceUID | report column (guess): Report
[POS] y=0.95 score=0.117
   MRI of left Knee with -Locator, SG PD FatSat, SG T2W FS, SG T1W, CO T2W FS, AX T2W FS * Irregular thining of intra-articular cartilage with spur formation and subchondral bone marrow edema/cysts , more prominent at lateral femoral condyle; tiny subchondral cyst in medial tibial plateau. * Wearing of articular cartilage of patella, major at medial facet and patellar ridge, reflecting grade 3 chondromalacia. Laxity of medial patellar retinaculum > The contour of ACL and MCL in a straight line and 

[POS] y=0.95 score=0.128
   MRI of left Knee with -Locator, SG PD FatSat, SG T2W FS, SG T1W, AX T2W FS, CO T2W FS > Linear signal intensity in posterior horn of medial meniscus, and unclear contour of meniscal root, in favor of te

## How to read the result
- **label_wrong** (report says the opposite of the label): the loss is the labeller's. A better model won't help; better labels or label-aware weighting will.
- **ambiguous** (report vague, finding probably borderline): little to gain.
- **real_miss** (report clearly states the finding, model disagrees): model-limited. Look at series chosen, slice coverage, vendor/site group, image quality for these.
- Change `LABEL` in cell 0 and re-run cells 5-6 for each label. Start with ACL, MCL, Medial Meniscus, Lateral Meniscus (model-limited per the spec) and Effusion (the known 15-case label mismatch).
Output files are in `/kaggle/working/error_analysis/` (commit with Save Version to keep them).